In [15]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedKFold
import torch

In [16]:
print("Torch :", torch.__version__)
print("CUDA :", torch.version.cuda)
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Torch : 2.11.0+cu128
CUDA : 12.8
CUDA Available : True
NVIDIA GeForce RTX 3060


In [17]:
DATA_DIR = Path("../data")

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

print(TRAIN_DIR)
print(TEST_DIR)

..\data\train
..\data\test


In [18]:
data = []

for class_dir in sorted(TRAIN_DIR.iterdir()):
    if not class_dir.is_dir():
        continue

    label = int(class_dir.name.split("_")[0])

    for img_path in class_dir.glob("*"):
        data.append({
            "filepath": str(img_path),
            "label": label,
            "class_name": class_dir.name
        })

df = pd.DataFrame(data)

print(df.shape)
df.head()

(24853, 3)


,filepath,label,class_name
0,..\data\train\0_Recyclable\R_1.jpg,0,0_Recyclable
1,..\data\train\0_Recyclable\R_10.jpg,0,0_Recyclable
2,..\data\train\0_Recyclable\R_100.jpg,0,0_Recyclable
3,..\data\train\0_Recyclable\R_1000.jpg,0,0_Recyclable
4,..\data\train\0_Recyclable\R_1001.jpg,0,0_Recyclable


In [19]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

df["fold"] = -1

for fold, (_, val_idx) in enumerate(skf.split(df, df["label"])):
    df.loc[val_idx, "fold"] = fold

In [20]:
for fold in range(5):
    print(f"\nFold {fold}")
    print(
        df[df.fold == fold]["class_name"]
        .value_counts()
    )


Fold 0
class_name
2_Organic       2291
0_Recyclable    1894
1_Electronic     786
Name: count, dtype: int64

Fold 1
class_name
2_Organic       2292
0_Recyclable    1893
1_Electronic     786
Name: count, dtype: int64

Fold 2
class_name
2_Organic       2292
0_Recyclable    1893
1_Electronic     786
Name: count, dtype: int64

Fold 3
class_name
2_Organic       2291
0_Recyclable    1893
1_Electronic     786
Name: count, dtype: int64

Fold 4
class_name
2_Organic       2291
0_Recyclable    1893
1_Electronic     786
Name: count, dtype: int64


In [21]:
df.to_csv("train_folds.csv", index=False)